In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import pandas as pd
import holidays
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
import matplotlib.pyplot as plt

In [ ]:
data2 = '/content/drive/MyDrive/Manhattan/weather/Data2/'

In [ ]:
def load_geohash_data(base_directory):
    geohash_data = {}

    # List all geohash directories in the base directory
    geohash_dirs = [name for name in os.listdir(base_directory) if os.path.isdir(os.path.join(base_directory, name))]

    for geohash in geohash_dirs:
        geohash_dir = os.path.join(base_directory, geohash)

        # Find all CSV files in the geohash directory
        csv_files = [f for f in os.listdir(geohash_dir) if f.endswith('.csv') ]

        for csv_file in csv_files:
            file_path = os.path.join(geohash_dir, csv_file)

            # Load the CSV file
            df = pd.read_csv(file_path)

            # Ensure the CSV has the correct columns
            if 'actual' in df.columns and 'predicted' in df.columns:
                actual_values = df['actual'].values
                predicted_values = df['predicted'].values

                # Store the data in the dictionary
                geohash_data[geohash] = (predicted_values, actual_values)
            else:
                print(f"Columns 'actual' and 'predicted' not found in {csv_file}.")
    return geohash_data

In [ ]:
base_directory = '/content/drive/MyDrive/Manhattan/weather/predictions/'  # Replace with the actual path
geohash_data = load_geohash_data(base_directory)

# To check the data
for geohash, data in geohash_data.items():
    print(f"Geohash: {geohash}")
    predicted, actual = data
    print(f"Predicted: {predicted[:5]}")  # Print first 5 values
    print(f"Actual: {actual[:5]}")

In [ ]:
for filename in os.listdir(base_directory):
    if filename.endswith(".csv"):
        geohash = filename.split('.')[0]

        df = pd.read_csv(os.path.join(base_directory, filename))

        geohash_dfs[geohash] = df

In [ ]:
geohash_data

In [ ]:
geohash_dfs = {}

for filename in os.listdir(data2):
    if filename.endswith(".csv"):
        geohash = filename.split('.')[0]

        df = pd.read_csv(os.path.join(data2, filename))

        geohash_dfs[geohash] = df

# Verify the loaded DataFrames
for geohash, df in geohash_dfs.items():
    print(f"Geohash: {geohash}")
    print(df.head())
    print("\n")

In [ ]:
def preprocess_data(df):
    trialdf = df[['OFFER_DATE', 'time_of_day', 'day_of_week', 'month', 'temperature', 'precipitation']]

    trialdf = trialdf.sort_values(by='OFFER_DATE')

    trialdf['OFFER_DATE'] = pd.to_datetime(trialdf['OFFER_DATE'])

    trialdf['next_OFFER_DATE'] = trialdf['OFFER_DATE'].shift(-1)

    trialdf['time_to_next_ride'] = (trialdf['next_OFFER_DATE'] - trialdf['OFFER_DATE']).dt.total_seconds()

    trialdf.drop(columns=['next_OFFER_DATE'], inplace=True)

    trialdf.dropna(subset=['time_to_next_ride'], inplace=True)

    us_holidays = holidays.US()
    trialdf['is_holiday'] = trialdf['OFFER_DATE'].dt.date.apply(lambda x: 1 if x in us_holidays else 0)

    filtered_df = trialdf.copy()

    return filtered_df

In [ ]:
filtered_dfs = {}
for geohash, df in geohash_dfs.items():
    filtered_df = preprocess_data(df)
    filtered_dfs[geohash] = filtered_df
    print(f"Geohash: {geohash}")
    print(filtered_df.head())
    print("\n")

In [ ]:
filtered_dfs.keys()

In [ ]:
def traintestsplit(df):
    # Select the features and target
    features = df[['time_of_day', 'day_of_week', 'month', 'temperature', 'precipitation', 'is_holiday']]
    target = df['time_to_next_ride']

    # Normalize the features
    feature_scaler = MinMaxScaler()
    features_scaled = feature_scaler.fit_transform(features)

    # Normalize the target
    target_scaler = MinMaxScaler()  # Initialize the scaler
    target_scaled = target_scaler.fit_transform(target.values.reshape(-1, 1))  # Fit and transform the target

    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(features_scaled, target_scaled, test_size=0.2, random_state=12)

    return (X_test, y_test), target_scaler

In [ ]:
test_data ={}
for geohash, df in filtered_dfs.items():
    test, target_scaler = traintestsplit(df)
    test_data[geohash] = (test, target_scaler)
    print(f"Geohash: {geohash}")
    print(test_data[geohash])
    print("\n")

In [ ]:
for geohash, (test, target_scaler) in test_data.items():
    print(f"Geohash: {geohash}")
    print(test[0],test[1], target_scaler)
    print("\n")

In [ ]:
model_dir = '/content/drive/MyDrive/Manhattan/weather/models/'

In [ ]:
def predict_model(geohash, X_test, y_test, target_scaler):
    model_path = os.path.join(model_dir, f"{geohash}_model.keras")
    model = tf.keras.models.load_model(model_path)

    # Print shape of X_test for debugging
    print(f"X_test shape before reshaping: {X_test.shape}")

    # Reshape X_test to ensure it is 2D
    if len(X_test.shape) == 1:  # If X_test is 1D
        X_test = X_test.reshape(1, -1)  # Reshape to (1, number of features)

    predictions = model.predict(X_test)

    # Ensure predictions and y_test are 2D for inverse_transform
    predictions = predictions.reshape(-1, 1)
    y_test = y_test.reshape(-1, 1)

    # Inverse transform the predictions and y_test
    predictions_inverse = target_scaler.inverse_transform(predictions)
    y_test_inverse = target_scaler.inverse_transform(y_test)

    predicted_val = predictions_inverse, y_test_inverse
    return predicted_val

In [ ]:
predicted_vals = {}
for geohash, (test, target_scaler) in test_data.items():
    predicted, actual = predict_model(geohash, test[0], test[1], target_scaler)
    predicted_vals[geohash] = (predicted, actual)
    print(f"Geohash: {geohash}")
    print(f"Predicted: {predicted}")
    print(f"Actual: {actual}")
    print("\n")

In [ ]:
import pandas as pd

# Assuming predicted_vals contains the predictions in the format {geohash: (predicted, actual)}
for geohash, (predicted, actual) in predicted_vals.items():
    # Convert the arrays into a DataFrame
    df = pd.DataFrame({
        'actual': actual.flatten(),  # Flatten to ensure it's a 1D array
        'predicted': predicted.flatten()  # Flatten to ensure it's a 1D array
    })

    # Define the filename based on the geohash
    filename = f"/content/drive/MyDrive/Manhattan/weather/predictions/{geohash}.csv"

    # Save the DataFrame to a CSV file
    df.to_csv(filename, index=False)

    print(f"Saved predictions to {filename}")

In [ ]:
import os
import pandas as pd

def load_geohash_data(base_directory):
    geohash_data = {}

    # List all geohash directories in the base directory
    geohash_dirs = [name for name in os.listdir(base_directory) if os.path.isdir(os.path.join(base_directory, name))]

    for geohash in geohash_dirs:
        geohash_dir = os.path.join(base_directory, geohash)

        # Load performance metrics
        metrics_file = os.path.join(geohash_dir, f"{geohash}_performance_metrics.csv")
        if os.path.exists(metrics_file):
            metrics_df = pd.read_csv(metrics_file)
        else:
            print(f"Metrics file not found for Geohash {geohash}.")
            continue

        # Assuming the actual and predicted values were also stored in the metrics CSV (as an example)
        # If they were stored in a different way, you would load them differently here
        if 'Actual' in metrics_df.columns and 'Predicted' in metrics_df.columns:
            actual_values = metrics_df['Actual'].values
            predicted_values = metrics_df['Predicted'].values
        else:
            print(f"Actual and Predicted columns not found in metrics for Geohash {geohash}.")
            continue

        # Store the data in the dictionary
        geohash_data[geohash] = (predicted_values, actual_values)

    return geohash_data


In [ ]:
def filter_outliers(actual, predicted, threshold=None, percentage=None):
    if threshold is not None:
        # Create a mask to filter out values greater than the threshold
        mask = actual <= threshold
    elif percentage is not None:
        # Calculate the percentage threshold based on actual values
        limit = np.percentile(actual, 100 - percentage)
        mask = actual <= limit
    else:
        # If neither threshold nor percentage is provided, return the original data
        return actual, predicted

    # Apply the mask to both actual and predicted values
    actual_filtered = actual[mask]
    predicted_filtered = predicted[mask]

    return actual_filtered, predicted_filtered

In [ ]:
def plot_results(predicted_vals, geohash, sample_size, plot_type='line', marker_color='blue', marker_size=10):
    # Check if the geohash is in the dictionary
    if geohash not in predicted_vals:
        print(f"Geohash {geohash} not found in the predictions dictionary.")
        return

    # Extract the predicted and actual values for the specified geohash
    predicted, actual = predicted_vals[geohash]

    # Flatten the arrays to 1D for plotting
    predicted = predicted.flatten()
    actual = actual.flatten()

    # Ensure the sample size is not greater than the filtered data size
    sample_size = min(sample_size, len(actual))

    # Calculate percentage error
    percentage_error = 100 * abs((actual - predicted) / actual)

    # Plot True Values vs Predictions
    plt.figure(figsize=(20, 15))
    if plot_type == 'line':
        plt.plot(actual[:sample_size], label='True Values', linestyle='-', linewidth=marker_size)
        plt.plot(predicted[:sample_size], label='Predictions', linestyle='-', linewidth=marker_size)
    elif plot_type == 'scatter':
        plt.scatter(range(sample_size), actual[:sample_size], label='True Values', c='blue', s=marker_size)
        plt.scatter(range(sample_size), predicted[:sample_size], label='Predictions', c=marker_color, s=marker_size)
    plt.legend()
    plt.xlabel('Sample Index')
    plt.ylabel('Time to Next Ride (s)')
    plt.title(f'True Values vs. Predictions for Geohash {geohash} (Sample Size: {sample_size})')
    plt.show()

    # Plot Percentage Error
    plt.figure(figsize=(20, 15))
    if plot_type == 'line':
        plt.plot(percentage_error[:sample_size], label='Percentage Error', linestyle='-', linewidth=marker_size)
    elif plot_type == 'scatter':
        plt.scatter(range(sample_size), percentage_error[:sample_size], label='Percentage Error', c=marker_color, s=marker_size)
    plt.legend()
    plt.xlabel('Sample Index')
    plt.ylabel('Percentage Error (%)')
    plt.title(f'Percentage Error for Geohash {geohash} (Sample Size: {sample_size})')
    plt.show()

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def calculate_performance_metrics(predicted_vals, geohash):
    # Check
    if geohash not in predicted_vals:
        print(f"Geohash {geohash} not found in the predictions dictionary.")
        return

    # Extract the predicted and actual values for the specified geohash
    predicted, actual = predicted_vals[geohash]

    # Flatten the arrays
    predicted = predicted.flatten()
    actual = actual.flatten()

    # Baseline predictions: using the mean of the actual values
    baseline_predicted = np.full_like(actual, np.mean(actual))

    # Calculate metrics for the model
    mae_model = mean_absolute_error(actual, predicted)
    mse_model = mean_squared_error(actual, predicted)
    rmse_model = np.sqrt(mse_model)


    # Calculate metrics for the baseline
    mae_baseline = mean_absolute_error(actual, baseline_predicted)
    mse_baseline = mean_squared_error(actual, baseline_predicted)
    rmse_baseline = np.sqrt(mse_baseline)

    return mae_model, mse_model, rmse_model, mae_baseline, mse_baseline, rmse_baseline

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_filtered_results_with_baseline(predicted_vals, geohash, sample_size, plot_type='line', marker_color='blue', marker_size=10, threshold=None, error_threshold=None, save_path=None):
    # Check if the geohash is in the dictionary
    if geohash not in predicted_vals:
        print(f"Geohash {geohash} not found in the predictions dictionary.")
        return

    # Extract the predicted and actual values for the specified geohash
    predicted, actual = predicted_vals[geohash]

    # Flatten the arrays to 1D for plotting
    predicted = predicted.flatten()
    actual = actual.flatten()

    # Filter outliers
    actual, predicted = filter_outliers(actual, predicted, threshold=threshold)

    # Check if the data has been filtered to empty
    if len(actual) == 0 or len(predicted) == 0:
        print(f"Filtered data is empty for Geohash {geohash} with sample size {sample_size} and threshold {threshold}.")
        return

    # Calculate baseline predictions using the mean of the actual values
    baseline_predicted = np.full_like(actual, np.mean(actual))

    # Calculate percentage error for predictions and baseline
    percentage_error_model = 100 * np.abs((actual - predicted) / actual)
    percentage_error_baseline = 100 * np.abs((actual - baseline_predicted) / actual)

    # Filter based on the error threshold
    if error_threshold is not None:
        valid_indices = percentage_error_model <= error_threshold
        actual = actual[valid_indices]
        predicted = predicted[valid_indices]
        baseline_predicted = baseline_predicted[valid_indices]
        percentage_error_model = percentage_error_model[valid_indices]
        percentage_error_baseline = percentage_error_baseline[valid_indices]

    # Re-adjust the sample size after error filtering
    sample_size = min(sample_size, len(actual))

    if len(actual) == 0 or len(predicted) == 0:
        print(f"Filtered data based on error threshold is empty for Geohash {geohash}.")
        return

    # Plot True Values, Predictions, and Baseline
    plt.figure(figsize=(20, 15))
    if plot_type == 'line':
        plt.plot(actual[:sample_size], label='True Values', linestyle='-', linewidth=marker_size)
        plt.plot(predicted[:sample_size], label='Predictions', linestyle='-', linewidth=marker_size, color=marker_color)
        plt.plot(baseline_predicted[:sample_size], label='Baseline', linestyle='--', linewidth=marker_size, color='green')
    elif plot_type == 'scatter':
        plt.scatter(range(sample_size), actual[:sample_size], label='True Values', c='blue', s=marker_size)
        plt.scatter(range(sample_size), predicted[:sample_size], label='Predictions', c=marker_color, s=marker_size)
        plt.scatter(range(sample_size), baseline_predicted[:sample_size], label='Baseline', c='green', s=marker_size)
    plt.legend()
    plt.xlabel('Sample Index')
    plt.ylabel('Time to Next Ride (s)')
    plt.title(f'True Values vs. Predictions vs. Baseline for Geohash {geohash} (Sample Size: {sample_size})')

    # Save or show the plot
    if save_path:
        plt.savefig(save_path)
    else:
        plt.show()
    plt.close()

    # Plot Percentage Error for Predictions and Baseline
    plt.figure(figsize=(20, 15))
    if plot_type == 'line':
        plt.plot(percentage_error_model[:sample_size], label='Percentage Error (Model)', linestyle='-', linewidth=marker_size, color=marker_color)
        # plt.plot(percentage_error_baseline[:sample_size], label='Percentage Error (Baseline)', linestyle='--', linewidth=marker_size, color='green')
    elif plot_type == 'scatter':
        plt.scatter(range(sample_size), percentage_error_model[:sample_size], label='Percentage Error (Model)', c=marker_color, s=marker_size)
        # plt.scatter(range(sample_size), percentage_error_baseline[:sample_size], label='Percentage Error (Baseline)', c='green', s=marker_size)
    plt.legend()
    plt.xlabel('Sample Index')
    plt.ylabel('Percentage Error (%)')
    plt.title(f'Percentage Error (Model vs. Baseline) for Geohash {geohash} (Sample Size: {sample_size})')

    # Save or show the plot
    if save_path:
        plt.savefig(save_path.replace('.png', '_error.png'))
    else:
        plt.show()
    plt.close()

def filter_outliers(actual, predicted, threshold=None):
    if threshold is None:
        return actual, predicted
    error = np.abs(actual - predicted)
    mask = error <= threshold
    return actual[mask], predicted[mask]

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

def save_geohash_analysis(predicted_vals, geohashes, plot_type='line', marker_color='blue', marker_size=10):
    sample_sizes = [100, 500, 1000]  # Define sample sizes
    thresholds = [3000, 2500, 2000]  # Define thresholds

    for geohash in geohashes:
        if geohash not in predicted_vals:
            print(f"Geohash {geohash} not found in the predictions dictionary.")
            continue

        # Create directory for the geohash
        directory = f"/content/drive/MyDrive/Manhattan/weather/predictions/Evaluation/{geohash}"
        os.makedirs(directory, exist_ok=True)

        metrics_list = []

        for sample_size in sample_sizes:
            for threshold in thresholds:
                plot_filename = f"{directory}/{geohash}_plot_samplesize_{sample_size}_threshold_{threshold}.png"
                plot_filtered_results_with_baseline(
                    predicted_vals=predicted_vals,
                    geohash=geohash,
                    sample_size=sample_size,
                    threshold=threshold,
                    plot_type=plot_type,
                    marker_color=marker_color,
                    marker_size=marker_size,
                    save_path=plot_filename
                )

                metrics = calculate_performance_metrics(predicted_vals, geohash)

                if metrics is not None:
                    mae_model, mse_model, rmse_model, mae_baseline, mse_baseline, rmse_baseline = metrics

                    # Store the metrics
                    metrics_list.append({
                        'Geohash': geohash,
                        'Sample Size': sample_size,
                        'Threshold': threshold,
                        'Model MAE': mae_model,
                        'Baseline MAE': mae_baseline,
                        'Model MSE': mse_model,
                        'Baseline MSE': mse_baseline,
                        'Model RMSE': rmse_model,
                        'Baseline RMSE': rmse_baseline,
                    })

        # Save metrics to a CSV file
        metrics_df = pd.DataFrame(metrics_list)
        metrics_filename = f"{directory}/{geohash}_performance_metrics.csv"
        metrics_df.to_csv(metrics_filename, index=False)

In [ ]:
plot_results(predicted_vals, 'dr5rus', 500, plot_type='line', marker_color='blue', marker_size=3)

In [ ]:
plot_filtered_results_with_baseline(predicted_vals, 'dr5rus', 100, plot_type='line',
                                    marker_color='red', marker_size=3,
                                    threshold=None, error_threshold=1000, save_path=None)

In [ ]:
calculate_performance_metrics(predicted_vals, 'dr5ru7', threshold = None, percentage = None)

In [ ]:
geohashes = ['dr5rus', 'dr5ru6', 'dr5ru4', 'dr5ruk', 'dr5rvp', 'dr5ru9',
       'dr5rud', 'dr5rvj', 'dr72h8', 'dr5ru2', 'dr5rsk', 'dr5rue',
       'dr5rgb', 'dr5ru7', 'dr5rum', 'dr5ru0', 'dr5ref', 'dr5rsp',
       'dr5rvn', 'dr5rsq', 'dr72j0', 'dr5ruq', 'dr5rsr', 'dr5ru5',
       'dr5ruu', 'dr5rez', 'dr5ru3', 'dr5rsj', 'dr5reg', 'dr5reu',
       'dr5rsn', 'dr5rsh', 'dr5ru1', 'dr5rsm', 'dr5ru8', 'dr5rug']  # List of geohashes to analyze

save_geohash_analysis(predicted_vals, geohashes, plot_type='line', marker_color='red', marker_size=2)